# PAI最終課題 — Genesis Go2 × ネコ外見 × Jev画像診断

正本: `AGENTS.md`

| 役割 | パス |
|---|---|
| 学習 | `scripts/train_go2.py` |
| 評価+録画+Jev | `scripts/eval_go2_record.py` |
| 環境 | `scripts/go2_env.py` |
| 共有ロジック | `src/cat_quad/` |

ランタイムは **G4 GPU**。学習 I/O は `/content/pai_work`、成果は Drive へ同期。

## 0. Drive マウントとパス

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

os.environ["PAI_DRIVE_ROOT"] = "/content/drive/MyDrive/pai-final-20261102"
os.environ["PAI_WORK_ROOT"] = "/content/pai_work"
os.environ["HF_HOME"] = "/content/pai_work/hf-cache"
RUN_ID = "20261102-0400-go2cat"
os.environ["RUN_ID"] = RUN_ID

Path(os.environ["PAI_DRIVE_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["PAI_WORK_ROOT"]).mkdir(parents=True, exist_ok=True)
print("DRIVE", os.environ["PAI_DRIVE_ROOT"])
print("WORK ", os.environ["PAI_WORK_ROOT"])
!nvidia-smi -L

## 1. 課題コードを `/content/pai_work/repo` へ

Drive に `pai-final-20261102/code/repo` がある場合はそれをコピー。無ければ GitHub clone に切り替え。

In [ ]:
import os, shutil, sys
from pathlib import Path

drive_repo = Path(os.environ["PAI_DRIVE_ROOT"]) / "code" / "repo"
work_repo = Path(os.environ["PAI_WORK_ROOT"]) / "repo"

if drive_repo.exists() and (drive_repo / "scripts" / "train_go2.py").is_file():
    if work_repo.exists():
        shutil.rmtree(work_repo)
    shutil.copytree(drive_repo, work_repo)
    print("copied", drive_repo, "->", work_repo)
else:
    print("Drive code/repo に train_go2.py がありません。")
    print("PC の 20261102-physical-ai-final を Drive .../code/repo に上げ直してください。")

assert (work_repo / "scripts" / "train_go2.py").is_file(), "train_go2.py missing"
sys.path.insert(0, str(work_repo / "src"))
sys.path.insert(0, str(work_repo / "scripts"))
%cd {work_repo}

## 2. Secrets（値は表示しない）

In [ ]:
from google.colab import userdata
import os

os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")
assert os.environ["TYPESAFE_API_KEY"], "TYPESAFE_API_KEY が空です"
print("TYPESAFE_API_KEY: set")
try:
    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
        print("HF_TOKEN: set")
except Exception:
    print("HF_TOKEN: skip")

## 3. 依存インストール

In [ ]:
!python scripts/setup_colab_env.py
!python scripts/sync_io.py bootstrap --run-id $RUN_ID

## 4. 学習（本体）

`scripts/train_go2.py` を実行。CKPT は `/content/pai_work/checkpoints/go2/$RUN_ID`、終了後 Drive 同期。

In [ ]:
!python scripts/train_go2.py --run-id $RUN_ID --num-envs 4096 --max-iterations 101 --sync-drive

## 5. 評価・録画・Jev（提出動画）

ネコ耳尻尾付きで headless 録画。`--with-jev` で画像診断して歩行指令を切替。

In [ ]:
!python scripts/eval_go2_record.py --run-id $RUN_ID --ckpt 100 --steps 400 --with-jev --sync-drive

## 6. 成果確認・最終同期

In [ ]:
import os
from pathlib import Path

work = Path(os.environ["PAI_WORK_ROOT"])
drive = Path(os.environ["PAI_DRIVE_ROOT"])
run = os.environ["RUN_ID"]
print("work ckpt", list((work / "checkpoints" / "go2" / run).glob("model_*.pt"))[:5])
print("work video", list((work / "outputs" / "videos").glob("*.mp4")))
print("drive video", list((drive / "exports" / "videos").glob("*.mp4")))
!python scripts/sync_io.py push-ckpt --run-id $RUN_ID
!python scripts/sync_io.py push-exports
!python scripts/sync_io.py push-hf-cache